# Experiment - Country encoding benchmark

> **Takeaway -** Group the ~180 countries into a structural region map (`guest_country_region`) - it ties the best encoding, is leakage-free, and scales to new markets; country is a top-tier predictor (rare-market cancel ~0.44 vs ~0.20 overall).

> **Decided -> `guest_country_region`.** On the temporal hold-out the structural
> region map tied the best encoding (HistGB **AUC 0.907 / AP 0.759**, within the
> bootstrap CI of `target_oof` and `mincount10`) and beat the old `topk5` trees by
> ~**0.08 AP**, with a rare-country-slice AUC of **0.97**. It is structural
> (leakage-free, lives in 00), the leanest multi-level option, and stays valid as
> new markets open. Implemented as `guest_country_region` in `src/features.py`
> (built in 00 §3.0); logged in `reports/open_decisions.md` 

> Rare-country bookings cancel at **0.439** vs **0.198** overall, so country is a
> *top-tier* feature and worth a SHAP/interaction check that it isn't just a proxy
> for non-refundable rate plans / OTA channel.

Controlled test: does HOW we collapse `primaryGuest_address_countryCode` change
predictive performance, and where should the transform live (00 vs. pipeline)?

**Design:**
- Split: temporal hold-out via `is_temporal_test`.
- Held fixed: the whole feature set except the country representation (we exclude
  both raw country AND `is_international` from the base, then add back exactly one
  country representation per arm, so the marginal value is isolated).
- Models: LogisticRegression (encoding-sensitive) + HistGradientBoosting (robust).
- Metrics: ROC AUC + Average Precision on the hold-out, overall and on the
  rare-country slice, with a bootstrap 95% CI on AUC.

Result is saved to `reports/tables/00_audit/country_encoding_benchmark.csv`.
> ⚠️ **LEAKAGE CAVEAT (2026-06-12):** This benchmark ran BEFORE the profile-leakage
> discovery: `primaryGuest_address_countryCode` is completed at/around check-in, so
> its missingness encodes the outcome (cancelled bookings stay empty; 38% of future
> bookings are empty at scoring time). The region-vs-drop gap (~0.09 AUC) was largely
> this leak, not structural signal — the honest value of country on profile-FILLED
> rows is ~+0.06 AUC. The encoding RANKING here (region wins among country
> representations) likely still holds, but the absolute numbers do not. See
> experiments/profile_leakage_quantification.ipynb and reports/open_decisions.md.


## What this experiment does
Guests' home countries span ~180 values, but most appear in only a handful of
bookings, meaning its too thin for a model to learn a reliable rule per country. I test how
to **group** countries so the real signal (where a guest is from clearly affects
cancellation) comes through without the model chasing noise.

In [1]:
import sys
from pathlib import Path
_here = Path.cwd().resolve()
while not (_here / "pyproject.toml").exists() and _here != _here.parent:
    _here = _here.parent
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

import time, warnings
import numpy as np
import pandas as pd

from src.data_loader import load_clean_reservations
from src.features import country_to_region, model_feature_roster  # region taxonomy + canonical feature roster

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore", category=UserWarning)  # keep benchmark output readable

COUNTRY_COL = "primaryGuest_address_countryCode"
TARGET_COL  = "status"   # int 0/1 (1 = Canceled) in the clean parquet


def _obj(frame) -> np.ndarray:
    """Pandas (nullable) -> 2D object ndarray with np.nan for missing, so
    sklearn's SimpleImputer / OneHotEncoder don't choke on pd.NA."""
    if isinstance(frame, pd.Series):
        frame = frame.to_frame()
    return frame.astype("string").to_numpy(dtype=object, na_value=np.nan)

In [2]:
def target_encode_oof(train_cat, y_train, test_cat, *, n_splits=5, smoothing=20.0, seed=42):
    """Leakage-safe target encoding: train values via K-fold out-of-fold means
    (no row sees its own target), test via the full-train map. Smoothing shrinks
    small categories toward the global mean; unseen -> global mean."""
    train_cat = train_cat.astype("string").fillna("__NA__").to_numpy()
    test_cat  = test_cat.astype("string").fillna("__NA__").to_numpy()
    y_train   = np.asarray(y_train, dtype=float)
    global_mean = y_train.mean()

    def _fit_map(cats, ys):
        d = pd.DataFrame({"c": cats, "y": ys})
        g = d.groupby("c")["y"].agg(["mean", "count"])
        smooth = (g["count"] * g["mean"] + smoothing * global_mean) / (g["count"] + smoothing)
        return smooth.to_dict()

    oof = np.full(len(train_cat), global_mean, dtype=float)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, va_idx in kf.split(train_cat):
        m = _fit_map(train_cat[tr_idx], y_train[tr_idx])
        oof[va_idx] = [m.get(c, global_mean) for c in train_cat[va_idx]]

    full = _fit_map(train_cat, y_train)
    test_vals = np.array([full.get(c, global_mean) for c in test_cat], dtype=float)
    return oof.reshape(-1, 1), test_vals.reshape(-1, 1)

In [3]:
def build_matrices(df_tr, df_te, y_tr, arm):
    """Fully-encoded train/test matrices for one arm. Fit on TRAIN only."""
    num_cols = [c for c in NUMERIC_BASE if c in df_tr.columns]
    cat_cols = [c for c in CATEGORICAL_BASE if c in df_tr.columns]

    num_imp = SimpleImputer(strategy="median").fit(df_tr[num_cols])
    scaler  = StandardScaler().fit(num_imp.transform(df_tr[num_cols]))
    Xtr = [scaler.transform(num_imp.transform(df_tr[num_cols]))]
    Xte = [scaler.transform(num_imp.transform(df_te[num_cols]))]

    cat_tr, cat_te = _obj(df_tr[cat_cols]), _obj(df_te[cat_cols])
    cat_imp = SimpleImputer(strategy="most_frequent").fit(cat_tr)
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(cat_imp.transform(cat_tr))
    Xtr.append(ohe.transform(cat_imp.transform(cat_tr)))
    Xte.append(ohe.transform(cat_imp.transform(cat_te)))

    ctr = df_tr[COUNTRY_COL].astype("string")
    cte = df_te[COUNTRY_COL].astype("string")

    if arm == "drop":
        pass
    elif arm == "is_international":
        Xtr.append((ctr.fillna("") != "DE").astype(float).to_numpy().reshape(-1, 1))
        Xte.append((cte.fillna("") != "DE").astype(float).to_numpy().reshape(-1, 1))
    elif arm in ("topk5", "mincount10", "region"):
        if arm == "topk5":
            keep = set(ctr.value_counts().head(5).index)
            tr_lvl = ctr.where(ctr.isin(keep), other="Other")
            te_lvl = cte.where(cte.isin(keep), other="Other")
        elif arm == "mincount10":
            vc = ctr.value_counts(); keep = set(vc[vc >= 10].index)
            tr_lvl = ctr.where(ctr.isin(keep), other="Other")
            te_lvl = cte.where(cte.isin(keep), other="Other")
        else:  # region: PRODUCTION taxonomy from src.features
            tr_lvl = ctr.map(country_to_region)
            te_lvl = cte.map(country_to_region)
        tr_lvl, te_lvl = tr_lvl.fillna("Other"), te_lvl.fillna("Other")
        ohe_c = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(_obj(tr_lvl))
        Xtr.append(ohe_c.transform(_obj(tr_lvl)))
        Xte.append(ohe_c.transform(_obj(te_lvl)))
    elif arm == "target_oof":
        te_tr, te_te = target_encode_oof(ctr, y_tr, cte)
        sc = StandardScaler().fit(te_tr)
        Xtr.append(sc.transform(te_tr)); Xte.append(sc.transform(te_te))
    else:
        raise ValueError(arm)
    return np.hstack(Xtr), np.hstack(Xte)

def bootstrap_auc_ci(y, p, n_boot=400, seed=42):
    rng = np.random.default_rng(seed); y = np.asarray(y); p = np.asarray(p); n = len(y)
    stats = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y[idx])) < 2: continue
        stats.append(roc_auc_score(y[idx], p[idx]))
    return np.percentile(stats, [2.5, 97.5])

In [4]:
# ---- load + temporal split + profile ----
df = load_clean_reservations()
assert {"is_temporal_test", COUNTRY_COL}.issubset(df.columns)
df = df.dropna(subset=[TARGET_COL]).copy()
NUMERIC_BASE, CATEGORICAL_BASE = model_feature_roster(df, exclude={"guest_country_region"})
print(f"base roster: {len(NUMERIC_BASE)} numeric + {len(CATEGORICAL_BASE)} categorical (country held out)")
y = df[TARGET_COL].astype(int).to_numpy()
test_mask = df["is_temporal_test"].astype(bool).to_numpy()
df_tr, df_te = df[~test_mask], df[test_mask]
y_tr, y_te = y[~test_mask], y[test_mask]

print(f"rows {len(df):,} | train {len(df_tr):,} | temporal-test {len(df_te):,}")
print(f"positive share - overall {y.mean():.3f} | train {y_tr.mean():.3f} | test {y_te.mean():.3f}")
s = df[COUNTRY_COL].astype("string"); vc = s.value_counts()
print(f"\n{COUNTRY_COL}: {s.nunique():,} unique | missing {s.isna().mean():.3f}")
print(f"coverage top5={vc.head(5).sum()/vc.sum():.3f} top10={vc.head(10).sum()/vc.sum():.3f}")
tr_country = df_tr[COUNTRY_COL].astype("string")
top5 = set(tr_country.value_counts().head(5).index)
rare_te = ~df_te[COUNTRY_COL].astype("string").isin(top5).to_numpy()
print(f"rare-country test rows (outside train top-5): {rare_te.sum():,} "
      f"({rare_te.mean():.1%}) | cancel rate {y_te[rare_te].mean():.3f}")

base roster: 18 numeric + 4 categorical (country held out)
rows 169,617 | train 127,212 | temporal-test 42,405
positive share - overall 0.207 | train 0.211 | test 0.198

primaryGuest_address_countryCode: 178 unique | missing 0.069
coverage top5=0.838 top10=0.892
rare-country test rows (outside train top-5): 8,115 (19.1%) | cancel rate 0.443


In [5]:
# ---- benchmark ----
arms = ["drop", "is_international", "topk5", "mincount10", "region", "target_oof"]
models = {
    "logreg": lambda: LogisticRegression(solver="saga", C=1.0, max_iter=1000, tol=1e-3),
    "histgb": lambda: HistGradientBoostingClassifier(max_depth=8, learning_rate=0.05,
                                                      max_iter=400, random_state=42),
}
rows = []
for arm in arms:
    Xtr, Xte = build_matrices(df_tr, df_te, y_tr, arm)
    for mname, factory in models.items():
        clf = factory().fit(Xtr, y_tr)
        p = clf.predict_proba(Xte)[:, 1]
        lo, hi = bootstrap_auc_ci(y_te, p)
        auc_rare = (roc_auc_score(y_te[rare_te], p[rare_te])
                    if rare_te.sum() > 30 and len(np.unique(y_te[rare_te])) == 2 else np.nan)
        rows.append({"arm": arm, "model": mname, "n_features": Xtr.shape[1],
                     "auc": roc_auc_score(y_te, p), "auc_ci_lo": lo, "auc_ci_hi": hi,
                     "ap": average_precision_score(y_te, p), "auc_rare": auc_rare})
        print(f"  {arm:16s} {mname:7s} feats={Xtr.shape[1]:4d} "
              f"AUC={rows[-1]['auc']:.4f} [{lo:.4f},{hi:.4f}] AP={rows[-1]['ap']:.4f} "
              f"rareAUC={auc_rare:.4f}")

res = pd.DataFrame(rows)
from src import tables_dir
out = tables_dir() / "00_audit" / "country_encoding_benchmark.csv"
out.parent.mkdir(parents=True, exist_ok=True)
res.to_csv(out, index=False)
for mname in models:
    print(f"\n[{mname}] sorted by hold-out AUC")
    sub = res[res.model == mname].sort_values("auc", ascending=False)
    print(sub[["arm","n_features","auc","auc_ci_lo","auc_ci_hi","ap","auc_rare"]]
          .to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"\nsaved -> {out}")

  drop             logreg  feats=  79 AUC=0.7417 [0.7361,0.7473] AP=0.4208 rareAUC=0.7196
  drop             histgb  feats=  79 AUC=0.7881 [0.7830,0.7938] AP=0.4827 rareAUC=0.7990
  is_international logreg  feats=  80 AUC=0.7830 [0.7779,0.7884] AP=0.4714 rareAUC=0.7219
  is_international histgb  feats=  80 AUC=0.8540 [0.8495,0.8584] AP=0.6114 rareAUC=0.8498
  topk5            logreg  feats=  85 AUC=0.8012 [0.7960,0.8063] AP=0.5023 rareAUC=0.7052
  topk5            histgb  feats=  85 AUC=0.8602 [0.8559,0.8647] AP=0.6307 rareAUC=0.8535
  mincount10       logreg  feats= 184 AUC=0.8618 [0.8572,0.8659] AP=0.6880 rareAUC=0.9472
  mincount10       histgb  feats= 184 AUC=0.8857 [0.8819,0.8896] AP=0.7251 rareAUC=0.9581
  region           logreg  feats=  86 AUC=0.8609 [0.8562,0.8650] AP=0.6923 rareAUC=0.9519
  region           histgb  feats=  86 AUC=0.8864 [0.8823,0.8904] AP=0.7287 rareAUC=0.9623
  target_oof       logreg  feats=  80 AUC=0.8596 [0.8550,0.8635] AP=0.6841 rareAUC=0.9443
  target_o

## Notes (resolved)

- **Decision: `region`** - ties the best encoding while being structural
  (leakage-free, lives in 00) and the leanest multi-level option. See
  `reports/open_decisions.md` #2 and the outcome box at the top.
- `max_iter` / `n_iter` are set for benchmark speed, so these AUCs are **relative**
  rankings between encodings - the final number comes from the tuned model in the
  rebuilt 01-04.